# Benchmarking LLMs on Multiple NLP Tasks

**Setup**

Run the cell below to import the necessary libraries, set the random seed for reproducibility, and check the GPU availability.




In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForQuestionAnswering
from transformers import pipeline
from datasets import load_dataset
from sklearn.metrics import precision_score, recall_score,  f1_score
import time
import random


# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



## Task Group 1 - Sentiment Analysis Benchmark

In this project, we'll be benchmarking different language models on standard NLP tasks. Let's start with sentiment analysis using a pre-trained model.

### Task 1

Set up the model and tokenizer for sentiment analysis using DistilBERT fine-tuned on the SST-2 dataset.

- Set the model name to `"distilbert-base-uncased-finetuned-sst-2-english"` and save it to the variable `MODEL_NAME_SENTIMENT`
- Use `AutoTokenizer` to load the tokenizer and save it to the variable `tokenizer_sentiment`
- Use `AutoModelForSequenceClassification` to load the model and save it to the variable `model_sentiment`
- Move the model to the GPU if available using the `device` variable
- Print a message confirming that the model and tokenizer are ready


In [ ]:
# Define the model for sentiment analysis (SST-2)
MODEL_NAME_SENTIMENT = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer_sentiment = AutoTokenizer.from_pretrained(MODEL_NAME_SENTIMENT)
model_sentiment = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_SENTIMENT)
model_sentiment.to(device)

print(f"Sentiment analysis model and tokenizer are ready.")


### Task 2

Next, we need a function to tokenize input text for the sentiment analysis model.

Create a function named `tokenize_sentiment_data` that processes input sentences for the model.
- The function should take three parameters: `sentence`, `tokenizer`, and `max_length` with a default value of `128`
- Create a variable named `inputs` by using the tokenizer to convert the sentence with padding and truncation
- Create a dictionary to move all tokenized inputs to the device and update the `inputs` variable
- Return the processed `inputs`

In [ ]:
def tokenize_sentiment_data(sentence, tokenizer, max_length=128):
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    return inputs


### Task 3

Now that we have a way to tokenize our data, let's create a function to calculate evaluation metrics for our model.

For binary classification tasks like sentiment analysis, accuracy alone isn't enough to evaluate performance. We also need to know how well our model identifies positive and negative sentiments specifically.

Create a function named `calculate_metrics` that computes precision and recall scores.

The function should take two parameters: `true_labels` and `predictions`

* `true_labels` are the actual labels from the dataset.
* `predictions` are the labels predicted by the model.
    
Calculate `precision` using sklearn's `precision_score` function

Calculate `recall` using sklearn's `recall_score` function

Return both `precision` and `recall` values

In [ ]:
def calculate_metrics(true_labels, predictions):
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    return precision, recall


### Task 4: 

Implement a function to evaluate the sentiment analysis model's performance across multiple evaluation metrics.

Define the function `evaluate_sentiment` that accepts three parameters: `model`, `tokenizer`, and `dataset`.
  
Activate evaluation mode on your model to ensure consistency in predictions by using `model.eval()`.

Initialize `correct` to `0` to track the number of correct predictions.

Initialize `total` to `0` to count all processed instances.

Create lists `predictions` and `true_labels` to store model outputs and actual labels from the dataset.

For each `item` in the dataset:
- Tokenize the sentence from `item['sentence']` using `tokenize_sentiment_data`, and store the result in `inputs`.
  
- Predict the sentiment by applying the model to the tokenized inputs and extract the prediction using `.argmax(-1).item()`.

  
- Append the prediction to the `predictions` list and the true label from `item['label']` to the `true_labels` list.

  
- Increment the `correct` counter if the prediction matches the true label.

After processing all items:

- Calculate `accuracy` as the ratio of `correct` predictions to `total` instances.
  
- Use the `calculate_metrics` function with `true_labels` and `predictions` to compute `precision` and `recall`.

  
- Return `accuracy`, `precision`, and `recall` from the function to provide insights into the model's effectiveness in sentiment analysis.



In [ ]:
def evaluate_sentiment(model, tokenizer, dataset):
    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0
    predictions = []
    true_labels = []
    
    for item in dataset:
        # Tokenize the sentence
        inputs = tokenize_sentiment_data(item['sentence'], tokenizer)
        
        # Get model predictions
        outputs = model(**inputs)
        prediction = outputs.logits.argmax(-1).item()
        
        # Collect predictions and true labels
        predictions.append(prediction)
        true_labels.append(item['label'])
        
        # Update correct and total counts for accuracy
        correct += (prediction == item['label'])
        total += 1
    
    # Calculate accuracy
    accuracy = correct / total
    
    # Calculate Precision and Recall
    precision, recall = calculate_metrics(true_labels, predictions)
    
    return accuracy, precision, recall


### Task 5

Now that we have defined the evaluation function, let's load the dataset and run the sentiment analysis benchmark.  

- Load the SST-2 dataset from the GLUE benchmark using `load_dataset`, specifying `"glue"` as the dataset name and `"sst2"` as the subset. Use the validation split for evaluation.  

- Define `sample_size` as `500` to limit the number of examples used for evaluation.  

- Generate a list of randomly selected indices using `random.sample()`, ensuring that the selected sample size does not exceed the number of available examples in `sentiment_dataset`.  

- Create a subset of the dataset by selecting only the randomly chosen samples using `select()`.  

- Evaluate the sentiment analysis model using the `evaluate_sentiment` function by passing `model_sentiment`, `tokenizer_sentiment`, and the sampled `sentiment_dataset`.  

- Print the sentiment analysis accuracy, precision, and recall using `print()`, ensuring each value is rounded to two decimal places.

In [ ]:
# Load the SST-2 dataset (sentiment analysis dataset from GLUE)
sentiment_dataset = load_dataset("glue", "sst2", split="validation")

# Sample a subset for faster evaluation (e.g., 500 samples)
sample_size = 500
sampled_indices = random.sample(range(len(sentiment_dataset)), min(sample_size, len(sentiment_dataset)))
sentiment_dataset = sentiment_dataset.select(sampled_indices)

# Evaluate the model
accuracy, precision, recall = evaluate_sentiment(model_sentiment, tokenizer_sentiment, sentiment_dataset)

print(f"Sentiment Analysis Accuracy: {accuracy:.2f}")
print(f"Sentiment Analysis Precision: {precision:.2f}")
print(f"Sentiment Analysis Recall: {recall:.2f}")


## Task Group 2

Now, let's start with the question answering benchmark using a pre-trained model.

### Task 6

Now let's start with the question-answering benchmark by setting up the model and tokenizer for the SQuAD dataset.  

- Set the model name to `"distilbert-base-uncased-distilled-squad"` and save it to the variable `MODEL_NAME_QNA`.  

- Use `AutoTokenizer` to load the tokenizer and save it to the variable `tokenizer_qna`.  

- Use `AutoModelForQuestionAnswering` to load the model and save it to the variable `model_qna`.  

- Move the model to the GPU if available using the `device` variable.  

- Print a message confirming that the model and tokenizer are ready.

In [ ]:
# Define the model for question answering (SQuAD)
MODEL_NAME_QNA = "distilbert-base-uncased-distilled-squad"
tokenizer_qna = AutoTokenizer.from_pretrained(MODEL_NAME_QNA)
model_qna = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME_QNA)
model_qna.to(device)

print(f"Question answering model and tokenizer are ready.")


### Task 7  

Now that we have the model ready, let's implement a function to evaluate the question-answering model's performance on the dataset.  

The function named `evaluate_qna` is defined, that takes three parameters: `model`, `tokenizer`, and `dataset`.  

The model is set to evaluation mode using `model.eval()` to ensure no updates occur during inference.   

- Initialize `correct` to `0` to track the number of correct exact matches and `total` to `0` to count the number of examples in the dataset.  

- Create empty lists `exact_matches` and `f1_scores` to store the exact match and F1 score results for each example.  

- Loop through each `item` in `dataset`, extracting the `question`, `context`, and `answers` fields.  

- Tokenize the `question` and `context` using `tokenizer`, ensuring padding, truncation, and a maximum sequence length of `512`, and store the output in `inputs`.  

- Move all tokenized inputs to the `device` for GPU processing.  

- Perform a forward pass with `model(**inputs)`, obtaining `start_logits` and `end_logits` to predict the start and end positions of the answer span.  

- Identify `start_idx` and `end_idx` by selecting the positions with the highest probability using `torch.argmax()` on `start_logits` and `end_logits`.  

- Convert the predicted token indices to text using `tokenizer.convert_ids_to_tokens()` and `tokenizer.convert_tokens_to_string()`, and store the output in `predicted_answer`.  

- Retrieve the correct answer from `answers['text'][0]` and normalize both `predicted_answer` and `true_answer` by stripping spaces and converting them to lowercase.  

- Compute `exact_match` by checking if `predicted_answer` exactly matches `true_answer` and append the result to `exact_matches`.  

- Compute the `F1` score using `f1_score()` from `sklearn` and append the result to `f1_scores`.  

- Update `correct` by adding `exact_match` and increment `total` by `1` for each example processed.  

- Compute `accuracy` by dividing the total number of correct exact matches by the total number of examples.  

- Compute `avg_exact_match` by averaging the values in `exact_matches` to get the proportion of predictions that exactly match the true answer.  

- Compute `avg_f1` by averaging the values in `f1_scores` to assess how well the predicted answer overlaps with the true answer.  

- Return `accuracy`, `avg_exact_match`, and `avg_f1` as the final evaluation metrics of the question-answering model.

In [ ]:
def evaluate_qna(model, tokenizer, dataset):
    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0
    exact_matches = []
    f1_scores = []
    
    for item in dataset:
        question = item['question']
        context = item['context']
        answers = item['answers']
        
        inputs = tokenizer(question, context, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {key: value.to(device) for key, value in inputs.items()}
        outputs = model(**inputs)
        start_score, end_score = outputs.start_logits, outputs.end_logits
        
        start_idx = torch.argmax(start_score)
        end_idx = torch.argmax(end_score)
        
        predicted_answer = tokenizer.convert_tokens_to_string(
            tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][start_idx:end_idx+1])
        )
        
        # Exact Match (EM)
        exact_match = int(predicted_answer.strip().lower() == answers['text'][0].strip().lower())
        exact_matches.append(exact_match)
        
        # F1 Score
        f1 = f1_score([answers['text'][0]], [predicted_answer], average='micro')
        f1_scores.append(f1)
        
        correct += exact_match
        total += 1
    
    accuracy = correct / total
    avg_f1 = sum(f1_scores) / len(f1_scores)
    avg_exact_match = sum(exact_matches) / len(exact_matches)
    
    return accuracy, avg_exact_match, avg_f1


### Task 8 

Now that we have implemented the evaluation function, let's load the dataset and benchmark the question-answering model.  

- Load the `SQuAD` dataset using `load_dataset` and specify the `validation` split to obtain the evaluation data.  

- Define the `sample_size` as `100` to limit the number of examples used for evaluation.  

- Generate a list of randomly selected indices using `random.sample()` to ensure the number of selected examples does not exceed the dataset length.  

- Create a subset of the dataset using `select()` with `sampled_indices` to work with a smaller sample for faster evaluation.  

- Evaluate the question-answering model using `evaluate_qna()` with `model_qna`, `tokenizer_qna`, and `qna_dataset`, computing accuracy, exact match, and F1 score.  

- Print the question-answering accuracy, exact match score, and F1 score rounded to two decimal places using `print()` in separate statements.

In [ ]:
# Load the SQuAD dataset (question answering dataset)
qna_dataset = load_dataset("squad", split="validation")

# Sample a subset for faster evaluation (e.g., 100 samples)
sample_size = 100
sampled_indices = random.sample(range(len(qna_dataset)), min(sample_size, len(qna_dataset)))
qna_dataset = qna_dataset.select(sampled_indices)

# Evaluate the model
qna_accuracy, qna_exact_match, qna_f1 = evaluate_qna(model_qna, tokenizer_qna, qna_dataset)

print(f"Question Answering Accuracy: {qna_accuracy:.2f}")
print(f"Question Answering Exact Match: {qna_exact_match:.2f}")
print(f"Question Answering F1 Score: {qna_f1:.2f}")


## Task Group 3
Now that we have implemented individual benchmarks for sentiment analysis and question answering, let's create a function to run both evaluations and consolidate the results.


### Task 9  

Define a function named `run_all_benchmarks` that evaluates both NLP tasks and stores the results.  

Create an empty dictionary `all_results` to store benchmark results for each task.  

Print `=== Running Sentiment Analysis Benchmark ===` to indicate the evaluation is starting.  

Call the `evaluate_sentiment` function with `model_sentiment`, `tokenizer_sentiment`, and `sentiment_dataset`, and store the returned values in `sentiment_accuracy`, `sentiment_precision`, and `sentiment_recall`.  

Store the sentiment analysis results in the `all_results` dictionary under the key `"sentiment_analysis"`, with sub-keys `"accuracy"`, `"precision"`, and `"recall"` storing their respective values.  

Print `=== Running Question Answering Benchmark ===` to indicate the evaluation is starting.  

Call the `evaluate_qna` function with `model_qna`, `tokenizer_qna`, and `qna_dataset`, and store the returned values in `qna_accuracy`, `qna_exact_match`, and `qna_f1`.  

Store the question-answering results in the `all_results` dictionary under the key `"question_answering"`, with sub-keys `"accuracy"`, `"exact_match"`, and `"f1_score"` storing their respective values.  

Return the `all_results` dictionary containing benchmark results for both tasks.

In [ ]:
def run_all_benchmarks():
    all_results = {}

    # Evaluate Sentiment Analysis
    print("\n=== Running Sentiment Analysis Benchmark ===")
    sentiment_accuracy, sentiment_precision, sentiment_recall = evaluate_sentiment(model_sentiment, tokenizer_sentiment, sentiment_dataset)
    all_results["sentiment_analysis"] = {
        "accuracy": sentiment_accuracy,
        "precision": sentiment_precision,
        "recall": sentiment_recall
    }

    # Evaluate Question Answering
    print("\n=== Running Question Answering Benchmark ===")
    qna_accuracy, qna_exact_match, qna_f1 = evaluate_qna(model_qna, tokenizer_qna, qna_dataset)
    all_results["question_answering"] = {
        "accuracy": qna_accuracy,
        "exact_match": qna_exact_match,
        "f1_score": qna_f1
    }

    return all_results




### Task 10

Let's run all benchmarks and display the results.  

Call the `run_all_benchmarks` function and store the returned results in the variable `benchmark_results`.   

Print the `benchmark_results` dictionary to view the evaluation metrics for sentiment analysis and question answering.

In [ ]:
# Run all benchmarks
benchmark_results = run_all_benchmarks()

# Print the results
print("\nBenchmark Results:")
print(benchmark_results)